# Enumerative Program Synthesis

We are given i_1, i_2, o_1 examples and our goal is to find the smallest program in this DSL that produces that produces o_1 for every example: 

```
E ::= x | y | 0 | 1 | 2 | + E E | - E E | * E E
```

For example, `- * x x y` means `x*x - y`

We want to enumerate every program in order of increasing size (number of tokens) returning the first one to satisfy the examples

## The synthesizer

Steps: 
1. Pull the expected output out of each example storing these in a tuple
2. Setup a dictionary for programs by size 
3. Setup num_enumerated counter and seen_outputs set (so we can skip programs like +x0 that is the same as x) 
4. Loop through program sizes 1 up to a "max size". For each size start an empty kept list. 
5. Generate every candidate of specified size. For size = 1, it gives the 5 terminals with their outputs. For larger sizes, it combines every operator with every pair of smaller programs whose sizes add up to size-1. The left and right side's outputs are already known, so we need to know the output of combining them with our selected operator.
6. Check each candidate as it is generated, adding 1 to num_enumerated. If its output equals the target then return the program and the count immediately. If its output is in seen outputs skip. Otherwise, add the candidate's output to seen_outputs and add the program to kept 
7. Store the kept programs in programs_by_size so they can be used as building blocks for larger sizes. 
8. If no size works and we hit max size return None


In [3]:
TERMINALS = ["x", "y", "0", "1", "2"]
OPERATORS = ["+", "-", "*"]

def apply_op(op, a, b):
    if op == "+":
        return a + b
    if op == "-":
        return a - b
    return a * b


def synthesize(examples, max_size=9):

    # goes through each example, pulls out the output, and stores them in a tuple for comparison later
    outputs_list = []
    for example in examples:
        i1, i2, o = example
        outputs_list.append(o)
    target = tuple(outputs_list)

    # dictionary mapping program size to a list of (program, outputs) pairs 
    programs_by_size = {}

    # every outputs tuple we've already seen 
    seen_outputs = set()

    # number of candidate programs we've enumerated so far (for reporting purposes)
    num_enumerated = 0

    # gets every program of a given size, along with its outputs on the examples 
    def candidates(size):
        # size 1: the terminals. x gives the first input of each example, y gives the second, and a constant gives its own value every time
        if size == 1:
            for t in TERMINALS:
                if t == "x":
                    outputs = tuple(i1 for (i1, _, _) in examples)
                elif t == "y":
                    outputs = tuple(i2 for (_, i2, _) in examples)
                else:
                    outputs = tuple(int(t) for _ in examples)
                yield t, outputs
            return

        # bigger sizes: build every program by combining smaller programs with an operator
        for op in OPERATORS:  # picks an operator first
            # the operator takes up 1 token, so the left and right sides share the remaining size - 1 tokens
            for left_size in range(1, size - 1):
                right_size = size - 1 - left_size

                # look up the stored programs that fit the left and right sizes 
                left_programs = programs_by_size.get(left_size, [])
                right_programs = programs_by_size.get(right_size, [])

                # we are going to nest the right side inside the left side, so we can get every combination of left and right programs 
                for left_pair in left_programs:  # every possible left side
                    left_prog = left_pair[0]     # the program string, e.g. "x"
                    left_out = left_pair[1]      # its outputs, e.g. (1, 4, 0)

                    for right_pair in right_programs:  # every possible right side
                        right_prog = right_pair[0]
                        right_out = right_pair[1]

                        # glue the pieces into one program string
                        program = op + " " + left_prog + " " + right_prog

                        # work out the new outputs one example at a time
                        outputs = []
                        for k in range(len(left_out)):
                            outputs.append(apply_op(op, left_out[k], right_out[k]))
                        outputs = tuple(outputs)

                        yield program, outputs

    for size in range(1, max_size + 1):
        kept = []
        for program, outputs in candidates(size):
            num_enumerated += 1

            # the first program that matches every example is the answer
            if outputs == target:
                return program, num_enumerated

            # pruning: skip programs that have same outputs as a program we've already seen
            if outputs in seen_outputs:
                continue
            seen_outputs.add(outputs)
            kept.append((program, outputs))

        programs_by_size[size] = kept

    # nothing up to max_size worked, so report failure
    return None, num_enumerated


# test
print(synthesize([(1, 2, 3), (4, 5, 9), (0, 7, 7)]))  # ("+ x y", 7)

('+ x y', 7)


## Tests and Answer Key

The tests folder has 20 test files. In the answer key txt file it lits the smallest program and how many programs the reference solution enumerated for each test. The goal of this cell is to run every test and check with the answer key. 

Note: test 20 can't be written in this DSL, so the right answer it report failure (None). The answer key's count is 53,906 which is the number of programs enumerated for a max size of 0 with pruning which is why the max size above defaults to 9. 

In [5]:
import time
from pathlib import Path

# turns the text of a test file into a list of (input1, input2, output) tuples
def parse_examples(text):
    examples = []
    for line in text.splitlines():
        line = line.strip()                           # remove extra spaces and newlines
        if line == "" or line.startswith("#"):        # skip blank lines and comments
            continue
        parts = line.split(",")                       # "1, 2, 3" → ["1", " 2", " 3"]
        i1, i2, o = int(parts[0]), int(parts[1]), int(parts[2])   # int() also ignores the spaces
        examples.append((i1, i2, o))
    return examples

# read the answer key: "test, intended solution, reference output, count (time)"
answer_key = {}
for line in Path("answer_key.txt").read_text().splitlines()[1:]:
    name, intended, rest = line.split(", ", 2)
    reference_program, count_info = rest.split("   (")
    reference_count = int(count_info.split()[0])
    answer_key[name] = (reference_program, reference_count)

passed = 0
for path in sorted(Path("tests").glob("*.txt")):
    exs = parse_examples(path.read_text())

    start = time.time()
    program, count = synthesize(exs)
    elapsed = time.time() - start

    expected_program, expected_count = answer_key[path.stem]
    # the key writes failure as the string "None"
    ok = str(program) == expected_program and count == expected_count
    passed += ok

    print(f"{path.stem:22} {program!s:20} enumerated={count:<6} "
          f"{'PASS' if ok else 'FAIL (expected ' + expected_program + ')'}  ({elapsed:.2f}s)")

print(f"\n{passed}/{len(answer_key)} tests match the answer key")

01_identity_x          x                    enumerated=1      PASS  (0.00s)
02_constant            2                    enumerated=5      PASS  (0.00s)
03_add                 + x y                enumerated=7      PASS  (0.00s)
04_sub                 - x y                enumerated=32     PASS  (0.00s)
05_double              + x x                enumerated=6      PASS  (0.00s)
06_mul                 * x y                enumerated=57     PASS  (0.00s)
07_add_const           + x 1                enumerated=9      PASS  (0.00s)
08_square              * x x                enumerated=56     PASS  (0.00s)
09_sum_plus_one        + x + y 1            enumerated=86     PASS  (0.00s)
10_neg                 - 0 x                enumerated=41     PASS  (0.00s)
11_triple              + x + x x            enumerated=81     PASS  (0.00s)
12_square_minus_y      - * x x y            enumerated=587    PASS  (0.00s)
13_xy_plus_x           + x * x y            enumerated=105    PASS  (0.00s)
14_sum_squar